### Import required Python packages

In [1]:
import pandas as pd
import xarray as xr
import s3fs
import pytest
import os
import sys
import numpy as np
import apache_beam as beam
import xarray_beam as xbeam
import datetime
import time

### Reorganize paths

In [2]:
module_dir = os.path.abspath('../src/main/')
sys.path.insert(0, module_dir)

### Load test datasets

In [ ]:
retrospective_data_2849991 = pd.read_csv(
    "../tests/test_data/retrospective_output_data_2849991.csv",
    index_col=0)
retrospective_data_6010106 = pd.read_csv(
    "../tests/test_data/retrospective_output_data_6010106.csv",
    index_col=0)
retrospective_data_942030011 = pd.read_csv(
    "../tests/test_data/retrospective_output_data_942030011.csv",
    index_col=0)

daily_retro_data_raw_2849991 = pd.read_csv(
    "../tests/test_data/streamflow_df_for_indices_computation_2849991.csv",
    index_col=0)

daily_retro_data_raw_6010106 = pd.read_csv(
    "../tests/test_data/streamflow_df_for_indices_computation_6010106.csv",
    index_col=0)

daily_retro_data_raw_942030011 = pd.read_csv(
    "../tests/test_data/streamflow_df_for_indices_computation_942030011.csv",
    index_col=0)

### Test the retrospective transformation using Beam

In [ ]:
from dataflow_nwm_retro_indices_transformation import FlattenZarrChunkFn

def get_sample_nwm_ds():
    nwm_retro_s3_path = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr'
    file_system = s3fs.S3FileSystem(anon=True)
    nwm_data_store = xr.open_zarr(file_system.get_mapper(nwm_retro_s3_path), consolidated=True)
    reach_ids_to_compare = [2849991, 6010106, 942030011]
    selected_nwm_data_store = nwm_data_store.sel(feature_id=reach_ids_to_compare)
    return selected_nwm_data_store

print("\nOriginal Dataset:")
print(get_sample_nwm_ds().isel(time=slice(0, 15)))

with beam.Pipeline() as pipeline:
    pipe1 = (pipeline | xbeam.DatasetToChunks(get_sample_nwm_ds().isel(time=slice(0, 15)), 
                                               chunks={'feature_id': 1, 'time': 5}))
    pipe2 = pipe1 | "Flattened Records" >> beam.ParDo(FlattenZarrChunkFn())
    pipe2 | "Print Flattened Records" >> beam.Map(print)



Original Dataset:


2026-02-21 10:46:20,729 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.


<xarray.Dataset> Size: 2kB
Dimensions:         (feature_id: 3, time: 15)
Coordinates:
    elevation       (feature_id) float32 12B dask.array<chunksize=(3,), meta=np.ndarray>
  * feature_id      (feature_id) int64 24B 2849991 6010106 942030011
    gage_id         (feature_id) |S15 45B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude        (feature_id) float32 12B dask.array<chunksize=(3,), meta=np.ndarray>
    longitude       (feature_id) float32 12B dask.array<chunksize=(3,), meta=np.ndarray>
    order           (feature_id) int32 12B dask.array<chunksize=(3,), meta=np.ndarray>
  * time            (time) datetime64[ns] 120B 1979-02-01T01:00:00 ... 1979-0...
Data variables:
    crs             |S1 1B ...
    qBtmVertRunoff  (time, feature_id) float64 360B dask.array<chunksize=(15, 1), meta=np.ndarray>
    qBucket         (time, feature_id) float64 360B dask.array<chunksize=(15, 1), meta=np.ndarray>
    qSfcLatRunoff   (time, feature_id) float64 360B dask.array<chunksize=(15, 1

2026-02-21 10:46:26,407 - WARNING - Dependencies required for Interactive Beam PCollection visualization are not available, please use: `pip install apache-beam[interactive]` to install necessary dependencies to enable all data visualization features.


2026-02-21 10:46:26,610 - INFO - Creating state cache with size 104857600


{'feature_id': 2849991, 'time': Timestamp('1979-02-01 01:00:00'), 'streamflow': 591.489990234375, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 210.77301025390625, 'qBucket': 0.004809999838471413, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 02:00:00'), 'streamflow': 590.9599609375, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 206.4910125732422, 'qBucket': 0.004819999914616346, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 03:00:00'), 'streamflow': 590.4400024414062, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 205.9610137939453, 'qBucket': 0.00482999999076128, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestamp('1979-02-01 04:00:00'), 'streamflow': 589.9299926757812, 'velocity': 0.3100000023841858, 'qBtmVertRunoff': 206.25601196289062, 'qBucket': 0.00484999967738986, 'qSfcLatRunoff': 0.0, 'q_lateral': 0.0}
{'feature_id': 2849991, 'time': Timestam

### Test converting hourly to daily data in Beam

In [ ]:
from dataflow_nwm_retro_indices_transformation import extract_date_key

retrospective_data_2849991_small = retrospective_data_2849991.iloc[:120, :].reset_index()
retrospective_data_2849991_small = retrospective_data_2849991_small.to_dict(orient='records')
retrospective_data_6010106_small = retrospective_data_6010106.iloc[:120, :].reset_index()
retrospective_data_6010106_small = retrospective_data_6010106_small.to_dict(orient='records')
retrospective_data_942030011_small = retrospective_data_942030011.iloc[:120, :].reset_index()
retrospective_data_942030011_small = retrospective_data_942030011_small.to_dict(orient='records')
retrospective_data_small = retrospective_data_2849991_small + retrospective_data_6010106_small + retrospective_data_942030011_small
with beam.Pipeline() as pipeline:
        daily_data = (pipeline 
                        | 'CreateInput' >> beam.Create(retrospective_data_small)
                        | 'ExtractDateKey' >> beam.Map(extract_date_key)
                        | 'AggregateDailyData' >> beam.combiners.Mean.PerKey()
                        | 'ReshuffleDaily' >> beam.Reshuffle())
        daily_data | 'PrintOutput' >> beam.Map(print)

        reach_grouped_data = (daily_data
                                | 'MapToKV' >> beam.Map(lambda KV: (KV[0][0], {'date': KV[0][1], 'streamflow': KV[1]}))
                                | 'GroupByFeatureID' >> beam.GroupByKey()
                                | 'PrintGroupedOutput' >> beam.Map(print))

2026-02-21 10:49:07,107 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
2026-02-21 10:49:07,214 - INFO - Creating state cache with size 104857600


((2849991, '1979-02-01'), 585.7886830205503)
((2849991, '1979-02-02'), 573.9337387084961)
((2849991, '1979-02-03'), 567.2212371826172)
((2849991, '1979-02-04'), 558.2916564941406)
((2849991, '1979-02-05'), 550.1874923706055)
((2849991, '1979-02-06'), 545.2899780273438)
((6010106, '1979-02-01'), 1896.6325577445652)
((6010106, '1979-02-02'), 1904.7799530029297)
((6010106, '1979-02-03'), 1909.6241251627605)
((6010106, '1979-02-04'), 1914.4699605305989)
((6010106, '1979-02-05'), 1915.4145253499348)
((6010106, '1979-02-06'), 1914.169921875)
((942030011, '1979-02-01'), 35.86695579860521)
((942030011, '1979-02-02'), 34.966665744781494)
((942030011, '1979-02-03'), 33.86999924977621)
((942030011, '1979-02-04'), 33.144165992736816)
((942030011, '1979-02-05'), 33.52458254496256)
((942030011, '1979-02-06'), 35.45000076293945)
(2849991, [{'date': '1979-02-01', 'streamflow': 585.7886830205503}, {'date': '1979-02-02', 'streamflow': 573.9337387084961}, {'date': '1979-02-03', 'streamflow': 567.22123718

### Test computing indices from daily data in Beam

In [ ]:
from dataflow_nwm_retro_indices_transformation import ComputeIndicesFn, SanitizeNaNsDoFn, indices_rowdict_schema

daily_retro_data_raw_2849991_records = daily_retro_data_raw_2849991.reset_index().to_dict(orient='records')
input_2849991 = [(2849991, daily_retro_data_raw_2849991_records)]
daily_retro_data_raw_6010106_records = daily_retro_data_raw_6010106.reset_index().to_dict(orient='records')
input_6010106 = [(6010106, daily_retro_data_raw_6010106_records)]
daily_retro_data_raw_942030011_records = daily_retro_data_raw_942030011.reset_index().to_dict(orient='records')
input_942030011 = [(942030011, daily_retro_data_raw_942030011_records)]
inputs = input_2849991 + input_6010106 + input_942030011
with beam.Pipeline() as pipeline:
        indices_dict = (
            pipeline 
            | 'CreateDailyData' >> beam.Create(inputs) 
            | 'ComputeIndices' >> beam.ParDo(ComputeIndicesFn())
            | 'SanitizeIndicesNaNs' >> beam.ParDo(SanitizeNaNsDoFn()).with_output_types(indices_rowdict_schema)
            | 'PrintIndices' >> beam.Map(print))
        
        

2026-02-21 10:49:50,413 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
2026-02-21 10:49:52,245 - INFO - Creating state cache with size 104857600
2026-02-21 10:49:52,501 - INFO - Reach ID 2849991: The complete daily streamflow dataset is loaded. Computation of indices starts...
100%|██████████| 1/1 [00:02<00:00,  2.68s/it]
2026-02-21 10:49:55,236 - INFO - Reach ID 2849991: All data processing and computation finished.
2026-02-21 10:49:55,273 - INFO - Reach ID 6010106: The complete daily streamflow dataset is loaded. Computation of indices starts...


{'monthwise_mean': [571.93, 731.25, 793.32, 616.48, 468.0, 355.14, 260.6, 196.57, 153.39, 126.6, 134.25, 303.94], 'monthwise_cov': [115.5, 100.64, 83.91, 59.93, 49.83, 45.73, 39.26, 34.15, 29.55, 25.78, 84.82, 121.79], 'nth_percentile_flows': [62.40416479110718, 77.80561479568482, 86.22991528511048, 101.34112278620402, 126.6792474746704, 139.29749727249146, 153.24833011627197, 237.8077023824056, 466.69134362538654, 833.4418553670247, 1181.086947886149, 2337.3612219238285, 6531.351908365886], 'variability_index': 0.3240433471759944, 'slope_fdc': 0.02403006739756811, 'flashiness_index': [0.06771681866189519, 0.06489289123300265, 0.06685558490787699, 0.057100836613097325, 41.84071038381231], 'sevenQ10': 71.01974424687327, 'mean_annual_7_day_min': 97.93485242636629, 'baseflow_index': 0.4528499908640328, 'zero_flow_days_n': 0.0, 'low_flow_days_n': 7.651162790697675, 'duration_low_flow_event': 21.933333333333334, 'high_flow_days_n': 4.767441860465116, 'duration_high_flow_event': 4.8809523809

100%|██████████| 1/1 [00:00<00:00, 20.34it/s]
2026-02-21 10:49:55,359 - INFO - Reach ID 6010106: All data processing and computation finished.
2026-02-21 10:49:55,395 - INFO - Reach ID 942030011: The complete daily streamflow dataset is loaded. Computation of indices starts...


{'monthwise_mean': [2535.23, 2533.41, 2894.2, 3268.36, 4040.58, 4295.92, 3710.29, 2897.15, 2702.43, 2699.67, 2517.99, 2587.83], 'monthwise_cov': [43.63, 39.1, 48.17, 48.22, 52.78, 54.64, 56.29, 47.32, 51.07, 62.73, 42.42, 46.87], 'nth_percentile_flows': [1213.9524688720703, 1420.2035450236003, 1518.1810251871746, 1662.6993779500326, 1895.9409545898436, 2015.4815254211426, 2107.001649983724, 2564.001200358073, 3536.534828186035, 5055.943918863931, 6121.454435221354, 9612.994578857424, 23111.973307291668], 'variability_index': 0.16655062113682473, 'slope_fdc': 0.010813743966680917, 'flashiness_index': [0.0304082689713012, 0.03345583078471513, 0.035392092733244424, 0.03394035044421549, 26.53094507932088], 'sevenQ10': 1426.2203535135245, 'mean_annual_7_day_min': 1885.4279704046405, 'baseflow_index': 0.8519715282134741, 'zero_flow_days_n': 0.0, 'low_flow_days_n': 0.0, 'duration_low_flow_event': 0, 'high_flow_days_n': 0.023255813953488372, 'duration_high_flow_event': 1.0, 'half_flow_date': (

100%|██████████| 1/1 [00:00<00:00, 13.81it/s]
2026-02-21 10:49:55,507 - INFO - Reach ID 942030011: All data processing and computation finished.


{'monthwise_mean': [81.89, 89.08, 114.27, 104.84, 160.18, 162.5, 92.82, 66.06, 74.21, 104.45, 97.97, 95.87], 'monthwise_cov': [109.02, 104.03, 108.55, 92.4, 139.9, 115.62, 114.5, 103.55, 144.35, 170.19, 135.26, 151.14], 'nth_percentile_flows': [18.831249952316284, 24.52472448825836, 27.468207633495332, 31.22445760567983, 37.89733241399129, 40.75510311126709, 43.579999144872026, 59.71479034423828, 108.06093494097391, 210.26457862854, 313.57809893290187, 729.0389485677091, 2747.714121500651], 'variability_index': 0.2907085052560591, 'slope_fdc': 0.018498671654567, 'flashiness_index': [0.2066203432332358, 0.21530279662674318, 0.25686373050766154, 0.23220913442812424, 16.333433053950465], 'sevenQ10': 22.8294757299419, 'mean_annual_7_day_min': 30.486148264981008, 'baseflow_index': 0.5377454693897243, 'zero_flow_days_n': 0.0, 'low_flow_days_n': 1.2093023255813953, 'duration_low_flow_event': 10.4, 'high_flow_days_n': 6.837209302325581, 'duration_high_flow_event': 3.5853658536585367, 'half_flo

### Test Local Implementation with ready retrospective data

In [7]:
retrospective_data = retrospective_data_2849991.to_dict(orient='records') + retrospective_data_6010106.to_dict(orient='records') + retrospective_data_942030011.to_dict(orient='records')

start = time.perf_counter()

with beam.Pipeline() as pipeline:
    dict_records = (
        pipeline
        | 'CreateInput' >> beam.Create(retrospective_data)
    )

    daily_data = (
        dict_records
        | 'ExtractDateKey' >> beam.Map(extract_date_key)
        | 'AggregateDailyData' >> beam.combiners.Mean.PerKey()
        )
        
    dict_records_sanitized = (
        dict_records 
        | 'SanitizeRetroNaNs' >> beam.ParDo(SanitizeNaNsDoFn()).with_output_types(indices_rowdict_schema))
        
    reach_grouped_data = (
        daily_data
        | 'MapToKV' >> beam.Map(lambda KV: (KV[0][0], {'date': KV[0][1], 'streamflow': KV[1]}))
        | 'GroupByFeatureID' >> beam.GroupByKey())
        
    indices_dict = (
        reach_grouped_data 
        | 'ComputeIndices' >> beam.ParDo(ComputeIndicesFn())
        | 'SanitizeIndicesNaNs' >> beam.ParDo(SanitizeNaNsDoFn()).with_output_types(indices_rowdict_schema))

    dict_records_sanitized | 'WriteRetroToLocal' >> beam.io.WriteToText(
            file_path_prefix="../test_data/local_implementation_outputs/retrospective_data_local_impl_1",
            file_name_suffix='.csv',
            shard_name_template='-SSSSS-of-NNNNN',
            num_shards=1
        )
    indices_dict | 'WriteIndicesToLocal' >> beam.io.WriteToText(
            file_path_prefix="../test_data/local_implementation_outputs/indices_data_local_impl_1",
            file_name_suffix='.csv',
            shard_name_template='-SSSSS-of-NNNNN',
            num_shards=1
        )
    
end = time.perf_counter()

elapsed = end - start
print(f'Time taken: {elapsed:.6f} seconds')

2026-02-21 10:50:18,385 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
2026-02-21 10:51:24,893 - INFO - Creating state cache with size 104857600
2026-02-21 10:52:08,109 - WARNING - Deleting 1 existing files in target path matching: -*-of-%(num_shards)05d
2026-02-21 10:52:08,172 - INFO - Reach ID GLOBAL: The complete daily streamflow dataset is loaded. Computation of indices starts...
100%|██████████| 1/1 [00:00<00:00, 20.83it/s]
2026-02-21 10:52:08,255 - INFO - Reach ID GLOBAL: All data processing and computation finished.
2026-02-21 10:52:08,275 - INFO - Starting finalize_write threads with num_shards: 1 (skipped: 0), batches: 1, num_threads: 1
2026-02-21 10:52:08,277 - INFO - Renamed 1 shards in 0.00 seconds.
2026-02-21 10:52:08,289 - WARNING - Deleting 1 existing files in target path matching: -*-of-%(num_shards)05d
2026-02-21 10:52:08,294 - INFO - Starting finalize_write threads with num_shards: 1 (skipped: 0), batches: 1, num_

Time taken: 110.073286 seconds


### Complete Local Implementation through Directrunner

In [ ]:
from dataflow_nwm_retro_indices_transformation import FlattenZarrChunkFn, retrospective_rowdict_schema

nwm_ds_3 = get_sample_nwm_ds()

def test_full_pipeline(chunking_dict, file_label):

    start = time.perf_counter()

    with beam.Pipeline() as pipeline:
        zarr_chunks_pcoll = (
            pipeline
            | 'SplitZarrIntoChunks' >> xbeam.DatasetToChunks(nwm_ds_3, chunks=chunking_dict)
            )
        dict_records = (
            zarr_chunks_pcoll
            | 'FlattenZarrChunk' >> beam.ParDo(FlattenZarrChunkFn())
            | 'ReshuffleAfterFlatten' >> beam.Reshuffle() 
            )
        dict_records_sanitized = (
            dict_records 
            | 'SanitizeRetroNaNs' >> beam.ParDo(SanitizeNaNsDoFn()).with_output_types(indices_rowdict_schema)
            )
        daily_data = (
            dict_records
            | 'ExtractDateKey' >> beam.Map(extract_date_key)
            | 'AggregateDailyData' >> beam.combiners.Mean.PerKey()
            | 'ReshuffleDaily' >> beam.Reshuffle()
            )
        reach_grouped_data = (
            daily_data
            | 'MapToKV' >> beam.Map(lambda KV: (KV[0][0], {'date': KV[0][1], 'streamflow': KV[1]}))
            | 'GroupByFeatureID' >> beam.GroupByKey()
            )
        indices_dict = (
            reach_grouped_data 
            | 'ComputeIndices' >> beam.ParDo(ComputeIndicesFn())
            | 'SanitizeIndicesNaNs' >> beam.ParDo(SanitizeNaNsDoFn()).with_output_types(indices_rowdict_schema)
            )
        dict_records_sanitized | 'WriteRetroToLocal' >> beam.io.WriteToText(
                file_path_prefix=f"../test_data/local_implementation_outputs/retrospective_data_{file_label}",
                file_name_suffix='.csv',
                shard_name_template='-SSSSS-of-NNNNN',
                num_shards=1
            )
        indices_dict | 'WriteIndicesToLocal' >> beam.io.WriteToText(
                file_path_prefix=f"../test_data/local_implementation_outputs/indices_data_{file_label}",
                file_name_suffix='.csv',
                shard_name_template='-SSSSS-of-NNNNN',
                num_shards=1
            )
        
    end = time.perf_counter()

    elapsed = end - start
    print(f'Time taken: {elapsed:.6f} seconds')

chunk_combinations = [
    {'time': 672, 'feature_id': 2000},
]

for run in range(len(chunk_combinations)):
    test_full_pipeline(chunking_dict=chunk_combinations[run], file_label=f"t{chunk_combinations[run]['time']}-f{chunk_combinations[run]['feature_id']}")

2026-02-21 11:28:29,741 - INFO - Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
2026-02-21 11:28:30,554 - INFO - Creating state cache with size 104857600
2026-02-21 17:29:46,981 - INFO - Starting finalize_write threads with num_shards: 1 (skipped: 0), batches: 1, num_threads: 1
2026-02-21 17:29:46,983 - INFO - Renamed 1 shards in 0.00 seconds.
2026-02-21 17:29:48,957 - INFO - Reach ID 2849991: The complete daily streamflow dataset is loaded. Computation of indices starts...
100%|██████████| 1/1 [00:00<00:00, 17.15it/s]
2026-02-21 17:29:49,056 - INFO - Reach ID 2849991: All data processing and computation finished.
2026-02-21 17:29:49,088 - INFO - Reach ID 6010106: The complete daily streamflow dataset is loaded. Computation of indices starts...
100%|██████████| 1/1 [00:00<00:00, 16.05it/s]
2026-02-21 17:29:49,193 - INFO - Reach ID 6010106: All data processing and computation finished.
2026-02-21 17:29:49,228 - INFO - Reach ID 942030011: The

Time taken: 15124.401486 seconds
